# Imports

In [1]:
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.model_selection import StratifiedKFold, cross_val_predict

## Utils

In [2]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [10]:
X_train = pd.read_parquet('../data/X_train_fe.parquet')
y_train = pd.read_parquet('../data/y_train.parquet')

X_test = pd.read_parquet('../data/X_test_fe.parquet')

In [11]:
X_train.head()

,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,...,g_sub_r,r_sub_i,i_sub_z,u_sub_r,u_sub_z,redshift_mult_r,redshift_mult_i,x_espatial_coordinates,y_espatial_coordinates,z_espatial_coordinates
id,,,,,,,,,,,,,,,,,,,,,
0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,...,1.537632,1.100813,0.636056,5.114196,6.851065,8.326031,7.875818,0.313090,0.024912,-0.949397
1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,...,1.499854,0.361141,0.439634,3.191300,3.992076,2.778353,2.721302,-0.408896,0.435262,0.802092
2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,...,-0.092712,0.589211,0.025263,-0.136637,0.477837,59.784407,58.120611,0.529713,0.466346,-0.708467
3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,...,2.032982,0.652096,0.450706,4.287302,5.390104,10.195392,9.845805,-0.116193,0.045921,-0.992165
4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,...,1.237231,0.335002,0.283262,3.468709,4.086973,10.134002,9.947821,-0.787468,-0.394539,0.473532


In [12]:
X_test.head()

,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,...,g_sub_r,r_sub_i,i_sub_z,u_sub_r,u_sub_z,redshift_mult_r,redshift_mult_i,x_espatial_coordinates,y_espatial_coordinates,z_espatial_coordinates
id,,,,,,,,,,,,,,,,,,,,,
577347,120.719779,23.924249,23.668066,21.951680,21.086183,20.180032,19.202124,0.429042,G/K,Red_Sequence,...,0.865497,0.906151,0.977908,2.581883,4.465942,9.046867,8.658089,0.081333,0.344971,-0.935083
577348,219.414419,42.171651,24.902933,22.338822,20.732163,19.860330,19.687691,0.867305,M,Red_Sequence,...,1.606658,0.871833,0.172640,4.170770,5.215243,17.981106,17.224961,-0.208809,0.113279,-0.971374
577349,173.568731,-1.756400,19.427591,18.474633,17.551314,16.570674,16.176765,0.224234,G/K,Blue_Cloud,...,0.923319,0.980640,0.393909,1.876277,3.250826,3.935608,3.715715,0.131045,0.129932,-0.982825
577350,184.903993,-1.411074,23.121029,21.526855,20.670159,20.417633,20.699095,0.066507,G/K,Red_Sequence,...,0.856696,0.252526,-0.281462,2.450870,2.421934,1.374717,1.357922,-0.143212,0.069175,-0.987272
577351,222.487816,15.381403,25.094282,22.643981,21.123173,19.439500,19.094158,0.977218,M,Red_Sequence,...,1.520808,1.683673,0.345342,3.971109,6.000123,20.641941,18.996626,0.799820,-0.507330,0.320787


In [13]:
for col in ["spectral_type", "galaxy_population"]:
    X_train[col] = X_train[col].astype("category")
    X_test[col] = X_test[col].astype("category")

# Machine Learning

In [43]:
models = dict(
    cat=load_pickle('../models/layer_1/model_catboost.pkl'),
    lgbm=load_pickle('../models/layer_1/model_lightgbm.pkl'),
    xgb=load_pickle('../models/layer_1/model_xgboost.pkl'),
    # hist=load_pickle('../models/layer_1/model_hist_gradient_boosting.pkl'),
    # extra=load_pickle('../models/layer_1/model_extra_tree.pkl'),
    # rf=load_pickle('../models/layer_1/model_random_forest.pkl'),
    # lda=load_pickle('../models/layer_1/model_lda.pkl'),
    # linear_svc=load_pickle('../models/layer_1/model_linear_svc.pkl'),
    # lg=load_pickle('../models/layer_1/model_logistic_regression.pkl'),
    # mlp=load_pickle('../models/layer_1/model_mlp.pkl'),
    # qda=load_pickle('../models/layer_1/model_qda.pkl'),
    # ridge=load_pickle('../models/layer_1/model_ridge.pkl'),
    # sgd=load_pickle('../models/layer_1/model_sgdclassifier.pkl'),
    # trunsvd_knn=load_pickle('../models/layer_1/model_trunsvd_knn.pkl'),
)

## Train Dataset

In [20]:
X_train_stacking = pd.DataFrame({})

In [27]:
X_train_stacking.head()

,cat_0,cat_1,cat_2
0,0.999798,0.000168,3.363467e-05
1,0.990474,0.000013,9.513256e-03
2,0.000009,0.999991,4.761647e-09
3,0.999951,0.000046,2.594706e-06
4,0.995128,0.004829,4.310274e-05


In [28]:
for model_name, model in tqdm(models.items()):
    
    print(f"Predicting Train Dataset {model_name}")

    predictions = cross_val_predict(
        model, 
        X_train, 
        y_train.class_encoded, 
        n_jobs=-1, 
        method='predict_proba', 
        cv=StratifiedKFold(shuffle=True, random_state=42, n_splits=5)
    )
    X_train_stacking[[f'{model_name}_0', f'{model_name}_1', f'{model_name}_2']] = predictions

  0%|                                                                                                                                                                                         | 0/2 [00:00<?, ?it/s]

Predicting Train Dataset lgbm


 50%|████████████████████████████████████████████████████████████████████████████████████████                                                                                        | 1/2 [11:44<11:44, 704.08s/it]

Predicting Train Dataset xgb


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [14:04<00:00, 422.23s/it]


## Test Dataset

In [44]:
X_test_stacking = pd.DataFrame({})

In [45]:
for model_name, model in models.items():
    
    print(f"Predicting Test Dataset {model_name}")
    
    X_test_stacking[[f'{model_name}_0', f'{model_name}_1', f'{model_name}_2']] = model.predict_proba(X_test)

Predicting Test Dataset cat
Predicting Test Dataset lgbm
Predicting Test Dataset xgb


# Saving

In [40]:
X_train_stacking.to_parquet('../data/X_train_stacking_layer_one.parquet')
X_test_stacking.to_parquet('../data/X_test_stacking_layer_one.parquet')

In [41]:
X_train_stacking.head()

,cat_0,cat_1,cat_2,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2
0,0.999798,0.000168,3.363467e-05,0.999950,0.000044,6.018048e-06,0.999949,0.000047,3.426932e-06
1,0.990474,0.000013,9.513256e-03,0.987437,0.000217,1.234521e-02,0.989377,0.000250,1.037301e-02
2,0.000009,0.999991,4.761647e-09,0.000010,0.999989,9.359197e-07,0.000008,0.999991,8.650742e-07
3,0.999951,0.000046,2.594706e-06,0.999908,0.000086,6.627740e-06,0.999958,0.000040,2.004308e-06
4,0.995128,0.004829,4.310274e-05,0.998499,0.001463,3.851883e-05,0.998851,0.001056,9.297417e-05


In [42]:
X_test_stacking.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2
0,0.999070,0.000891,0.000039,0.998518,0.000901,5.804351e-04
1,0.998915,0.001079,0.000006,0.999453,0.000546,8.148580e-07
2,0.998171,0.000653,0.001176,0.999653,0.000099,2.479587e-04
3,0.002765,0.000630,0.996606,0.001739,0.000391,9.978704e-01
4,0.999806,0.000179,0.000015,0.999813,0.000187,6.986474e-07


In [34]:
X_train.shape

(577347, 21)

In [35]:
X_test.shape

(247435, 21)

In [36]:
y_train.head()

,class,class_encoded
id,,
0,GALAXY,0
1,GALAXY,0
2,QSO,1
3,GALAXY,0
4,GALAXY,0
